In [ ]:
# Imports and Spark session 
from pyspark.sql import SparkSession, functions as F
import os

spark = SparkSession.builder.appName("NutritionalOutlierAnalysis").getOrCreate()
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/15 18:11:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/15 18:11:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.0.1


In [2]:
# Configuration
input_path = "../output/nutritional_profiles"
fallback_file = "../output/nutritional_profiles/part-00000-d5d8abf9-0202-406e-af46-ff13446ce22f-c000.snappy.parquet"
output_dir = "../output/NutritionalOutlierAnalysis"
os.makedirs(output_dir, exist_ok=True)

# Strict schema mapping (edit as needed for your dataset)
schema_columns = {
    'id': 'fdc_id',
    'name': 'food_description',
    'category': 'food_type',
    'energy': 'energy',
    'protein': 'protein',
    'carb': 'carbs',
    'fat': 'total_fat',
    'fiber': 'fiber',
    'sugar': 'sugars',
    'saturated_fat': 'saturated_fat',
    'cholesterol': 'cholesterol',
    'sodium': 'sodium',
    'calcium': 'calcium',
    'iron': 'iron',
    'potassium': 'potassium',
    'vitamin_c': 'vitamin_c',
    'vitamin_a': 'vitamin_a',
    'vitamin_b12': 'vitamin_b12',
}

nutrient_cols = [
    'energy', 'protein', 'carb', 'fat', 'fiber', 'sugar', 'saturated_fat',
    'cholesterol', 'sodium', 'calcium', 'iron', 'potassium', 'vitamin_c', 'vitamin_a', 'vitamin_b12'
]

In [ ]:
# Data loading with strict schema 
from pyspark.sql.types import DoubleType

try:
    df = spark.read.parquet(input_path)
except Exception:
    df = spark.read.parquet(fallback_file)

for std_col, orig_col in schema_columns.items():
    if orig_col in df.columns:
        df = df.withColumnRenamed(orig_col, std_col)

selected_cols = ['id', 'name', 'category'] + nutrient_cols
df = df.select(*[c for c in selected_cols if c in df.columns])

for colname in nutrient_cols:
    if colname in df.columns:
        df = df.withColumn(colname, df[colname].cast(DoubleType()))

df.cache()
df.show(5)

+-------+--------------------+------------+------+-------+-----+-----+-----+-----+-------------+-----------+------+-------+----+---------+---------+---------+-----------+
|     id|                name|    category|energy|protein| carb|  fat|fiber|sugar|saturated_fat|cholesterol|sodium|calcium|iron|potassium|vitamin_c|vitamin_a|vitamin_b12|
+-------+--------------------+------------+------+-------+-----+-----+-----+-----+-------------+-----------+------+-------+----+---------+---------+---------+-----------+
| 408259|HERDEZ, SALSA CAS...|branded_food|  32.0|    0.0| 6.45|  0.0|  0.0| 3.23|          0.0|        0.0| 871.0|    0.0| 0.0|     NULL|      7.7|     NULL|       NULL|
| 467562|          BEEF JERKY|branded_food| 357.0|  67.86| 7.14| 7.14|  3.6|  0.0|         3.57|      125.0|3429.0|   71.0|9.64|     NULL|      0.0|     NULL|       NULL|
| 484195|ROCKET PISS BITTE...|branded_food|  45.0|    0.0|11.27|  0.0| NULL|10.99|         NULL|       NULL|   7.0|   NULL|NULL|     NULL|     NU

In [ ]:
# Outlier detection using IQR for each nutrient 
def get_outliers_spark(df, col):
    quantiles = df.approxQuantile(col, [0.25, 0.75], 0.01)
    if len(quantiles) < 2:
        return None, None, None, None
    Q1, Q3 = quantiles
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    high_outliers = df.filter(df[col] > upper).select('id', 'name', 'category', col)
    low_outliers = df.filter(df[col] < lower).select('id', 'name', 'category', col)
    return lower, upper, high_outliers, low_outliers

outlier_results = {}
for nutrient in nutrient_cols:
    if nutrient in df.columns:
        lower, upper, high, low = get_outliers_spark(df, nutrient)
        outlier_results[nutrient] = {
            'lower': lower,
            'upper': upper,
            'high': high,
            'low': low
        }
        print(f"Nutrient: {nutrient}")
        print(f"  Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")
        print(f"  High outliers: {high.count() if high else 0}, Low outliers: {low.count() if low else 0}")

Nutrient: energy
  Lower bound: -367.00, Upper bound: 849.00
  High outliers: 40, Low outliers: 0
Nutrient: protein
  Lower bound: -13.95, Upper bound: 24.37
  High outliers: 237, Low outliers: 0
Nutrient: carb
  Lower bound: -70.53, Upper bound: 135.34
  High outliers: 8, Low outliers: 0
Nutrient: fat
  Lower bound: -26.79, Upper bound: 44.65
  High outliers: 225, Low outliers: 0
Nutrient: fiber
  Lower bound: -5.40, Upper bound: 9.00
  High outliers: 253, Low outliers: 0
Nutrient: sugar
  Lower bound: -30.73, Upper bound: 54.36
  High outliers: 310, Low outliers: 0
Nutrient: saturated_fat
  Lower bound: -10.47, Upper bound: 17.45
  High outliers: 242, Low outliers: 0
Nutrient: cholesterol
  Lower bound: -34.50, Upper bound: 57.50
  High outliers: 516, Low outliers: 0
Nutrient: sodium
  Lower bound: -732.00, Upper bound: 1324.00
  High outliers: 300, Low outliers: 0
Nutrient: calcium
  Lower bound: -151.50, Upper bound: 252.50
  High outliers: 337, Low outliers: 0
Nutrient: iron
  Low

In [ ]:
# Visualization: Print top N outliers for a selected nutrient 
def print_outliers(nutrient, outlier_type='high', top_n=10):
    if nutrient not in outlier_results:
        print(f"Nutrient '{nutrient}' not found.")
        return
    sdf = outlier_results[nutrient][outlier_type]
    if sdf is None or sdf.count() == 0:
        print(f"No {outlier_type} outliers for {nutrient}.")
        return
    order_col = F.desc(nutrient) if outlier_type == 'high' else F.asc(nutrient)
    sdf.orderBy(order_col).limit(top_n).show(truncate=False)

# Example: Print high sodium outliers
print_outliers('sodium', 'high', top_n=10)
print_outliers('protein', 'high', top_n=10)
print_outliers('carb', 'low', top_n=10)
print_outliers('fat', 'high', top_n=10)
print_outliers('fiber', 'low', top_n=10)


+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+
|id     |name                                                                                                                                                                                                                       |category    |sodium   |
+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+
|1709731|BUTCHER'S RESERVE STEAK SAUCE                                                                                                                                                                                              |branded_food

In [ ]:
# Example recommendation logic for special dietary needs 
def recommend_foods_for_deficiency(nutrient, top_n=10):
    if nutrient not in outlier_results:
        print(f"Nutrient '{nutrient}' not found.")
        return
    sdf = outlier_results[nutrient]['high']
    if sdf is None or sdf.count() == 0:
        print(f"No high outliers for {nutrient}.")
        return
    print(f"Top {top_n} foods for {nutrient} deficiency:")
    sdf.orderBy(F.desc(nutrient)).limit(top_n).show(truncate=False)
    return sdf.orderBy(F.desc(nutrient)).limit(top_n)

def recommend_foods_for_restriction(nutrient, top_n=10):
    if nutrient not in outlier_results:
        print(f"Nutrient '{nutrient}' not found.")
        return
    sdf = outlier_results[nutrient]['low']
    if sdf is None or sdf.count() == 0:
        print(f"No low outliers for {nutrient}.")
        return
    print(f"Top {top_n} foods for {nutrient} restriction:")
    sdf.orderBy(F.asc(nutrient)).limit(top_n).show(truncate=False)
    return sdf.orderBy(F.asc(nutrient)).limit(top_n)

# Example: Recommend foods for iron deficiency
recommend_foods_for_deficiency('iron', top_n=10)
# Example: Recommend foods for sodium restriction
recommend_foods_for_restriction('sodium', top_n=10)
recommend_foods_for_restriction('fat', top_n=10)
recommend_foods_for_restriction('carb', top_n=10)
recommend_foods_for_deficiency('protein', top_n=10)
recommend_foods_for_deficiency('fiber', top_n=10)
recommend_foods_for_deficiency('potassium', top_n=10)


Top 10 foods for iron deficiency:
+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------+
|id     |name                                                                                                                                                                                                                       |category    |iron   |
+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------+
|2326587|THAI ME OVER KETOGENIC SAVORY BITE, THAI ME OVER                                                                                                                                                            

DataFrame[id: bigint, name: string, category: string, potassium: double]

In [ ]:
# Save outlier results and recommendations to output directory 
for nutrient, res in outlier_results.items():
    for outlier_type in ['high', 'low']:
        sdf = res[outlier_type]
        if sdf is not None and sdf.count() > 0:
            out_path_csv = f"{output_dir}/{nutrient}_{outlier_type}_outliers.csv"
            sdf.write.mode('overwrite').csv(out_path_csv, header=True)
            print(f"Saved {outlier_type} outliers for {nutrient} to {out_path_csv} ({sdf.count()} rows)")

# Example: Save iron deficiency and sodium restriction recommendations 
iron_high = outlier_results['iron']['high']
sodium_low = outlier_results['sodium']['low']
if iron_high is not None and iron_high.count() > 0:
    iron_high.write.mode('overwrite').csv(f"{output_dir}/iron_deficiency_recommendations.csv", header=True)
    print(f"Saved iron deficiency recommendations to {output_dir}/iron_deficiency_recommendations.csv ({iron_high.count()} rows)")
if sodium_low is not None and sodium_low.count() > 0:
    sodium_low.write.mode('overwrite').csv(f"{output_dir}/sodium_restriction_recommendations.csv", header=True)
    print(f"Saved sodium restriction recommendations to {output_dir}/sodium_restriction_recommendations.csv ({sodium_low.count()} rows)")


Saved high outliers for energy to ../output/NutritionalOutlierAnalysis/energy_high_outliers.csv (40 rows)
Saved high outliers for protein to ../output/NutritionalOutlierAnalysis/protein_high_outliers.csv (237 rows)
Saved high outliers for carb to ../output/NutritionalOutlierAnalysis/carb_high_outliers.csv (8 rows)
Saved high outliers for fat to ../output/NutritionalOutlierAnalysis/fat_high_outliers.csv (225 rows)
Saved high outliers for fiber to ../output/NutritionalOutlierAnalysis/fiber_high_outliers.csv (253 rows)
Saved high outliers for sugar to ../output/NutritionalOutlierAnalysis/sugar_high_outliers.csv (310 rows)
Saved high outliers for saturated_fat to ../output/NutritionalOutlierAnalysis/saturated_fat_high_outliers.csv (242 rows)
Saved high outliers for cholesterol to ../output/NutritionalOutlierAnalysis/cholesterol_high_outliers.csv (516 rows)
Saved high outliers for sodium to ../output/NutritionalOutlierAnalysis/sodium_high_outliers.csv (300 rows)
Saved high outliers for calc